# 06 · Line profiles and optical depth

For every line j the gas at x absorbs the photon labelled by the observer's line-centred velocity v_† at the local
wavelength λ'_j = λ_j e^{v_†/c} a(x)/a_† D⁻¹(x) (review notes, Eq. velocity_units_normalized), i.e. at the
frequency offset ν_j = c(1/λ' − 1/λ_j). The profile is a Voigt function φ = V(ν; σ, γ) with
σ = b/(√2 λ_j), b = √(2k_B T/m) + b_turb and γ = (A_21 + A_coll)/4π (Eq. line_profile), and

  τ_j(v_†) = C_j f_j Σ_i w_i n_com,i a_i⁻² φ_ij(v_†),  F = e^{−τ}  (Eq. tau_straight_los, discretised).

Runtime: ~1 min.

**Requires:** lines (01), skewers (03, a(x) from 02) and fields (04); the last section also simulation outputs (05).  
**Independent of:** the summary statistics (07).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, use_backend, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM,
                                      CMAPS, LABELS, FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)
use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # dark style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

In [ ]:
plot_dag(('tau',)); plt.show()

In [ ]:
from skewerjax.core import (LineBundle, LINES, voigt as V, line_wavelength_grid_A, frequency_offset_GHz, line_profiles,
                            compute_optical_depth, los_samples, resolution_report, wrap_to_ref_line, SkewerBundle, Fields,
                            ScalarField, VectorField, LogRamp, VelocityRamp, Constant, plane_points, analytic as an, phys)
lines = LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"], ref_line_id="HI-LyA")
print(f"{len(LINES)} transitions in lines_table.dat")
for l, dv in zip(lines.lines, np.asarray(lines.delta_v)):
    print(f"  {l.id:11s} lambda0 = {l.lambda0_A:9.4f} A  f = {l.f_osc:.4f}  A21 = {l.A21_Hz:.3e} Hz  m = {l.mass_amu:7.3f} amu  delta_v = {dv:9.1f} km/s")

## 1 · The Voigt function

Three implementations of the same V(x; σ, γ) = (G_σ ∗ L_γ)(x), the convolution of a Gaussian of standard deviation σ
and a Lorentzian of half-width γ: `"ExoJAX"` (default; a Humlíček-type approximation of the Faddeeva function,
jit/grad), `"convolution"` (G ∗ L by FFT on the grid, refined 4×; it needs a grid that resolves both widths) and
`"scipy_wofz"` (V = Re w(z)/(√(2π) σ), z = (x + iγ)/(√2 σ), NumPy, the reference). The four (σ, γ) pairs combine
10⁻³ and 1: the Gaussian limit (γ ≪ σ), the Lorentzian limit (σ ≪ γ), and two Voigt profiles with both widths
equal.

In [ ]:
from skewerjax.utils.plotting import plot_voigt_limits
x = jnp.linspace(-9, 9, 2 ** 13)
sigma = np.array([1e-3, 1.0, 1e-3, 1.0]); gamma = np.array([1e-3, 1e-3, 1.0, 1.0])
curves = {"scipy_wofz": np.stack([V.voigt_scipy(np.asarray(x), s, g) for s, g in zip(sigma, gamma)]),
          "ExoJAX": np.stack([np.asarray(V.voigt_exojax(x, s, g)) for s, g in zip(sigma, gamma)]),
          "convolution (FFT, oversample 4)": np.stack([np.asarray(V.voigt_convolution(x, s, g, oversample=4)) for s, g in zip(sigma, gamma)])}
fig, axs = plot_voigt_limits(np.asarray(x), sigma, gamma, curves)
add_panel_letters(axs); fig.suptitle("Voigt: the Gaussian and Lorentzian limits; ExoJAX (solid) and FFT convolution (dashed) on scipy's wofz", fontsize=21)
plt.show()
for name in ("ExoJAX", "convolution (FFT, oversample 4)"):
    r = np.abs(curves[name] / curves["scipy_wofz"] - 1)
    core = np.abs(np.asarray(x))[None] < 3 * np.maximum(sigma, gamma)[:, None]
    print(f"{name:32s} max |V / wofz - 1| per pair (|x| < 3 max(sigma, gamma)):", np.array2string(np.where(core, r, 0).max(1), precision=1))

ExoJAX agrees with wofz to ~10⁻⁶ or better for every pair. The convolution is exact (10⁻¹⁴) when the grid
resolves both widths, and fails where it cannot: σ = γ = 10⁻³ is far below the grid spacing (2.2×10⁻³), and the
Lorentzian wings are cut at the grid ends (`scripts/validation/check_voigt.py`).

## 2 · Line profiles at points

`line_profiles(events, directions, v, a_†, lines, fields)` evaluates φ_ij for every line and point: a hotter gas
broadens the lines, a peculiar velocity along the photon shifts them (to larger v for gas moving towards the
observer... in the frame of the photon direction).

In [ ]:
pts = plane_points(1e-3, "y", 5e-4, 1, 5)[0]
events = jnp.concatenate([jnp.ones((5, 1)), pts], -1)
v = jnp.linspace(-400, 400, 801)
fig, axs = plt.subplots(1, 2, figsize=(20, 6), layout="constrained")
for ax, (Tlo, Thi, vlo, vhi, title) in zip(axs, ((1e3, 1e6, 0.0, 0.0, r"$T$ from $10^3$ to $10^6$ K"), (1e4, 1e4, -300.0, 300.0, r"$v_x$ from −300 to +300 km/s, $T = 10^4$ K"))):
    sp = {"temperature": ScalarField(LogRamp(Tlo, Thi, 1e-3, 3)), "velocity": VectorField(VelocityRamp(vlo, vhi, 1e-3)),
          "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0)), "number_density": ScalarField(Constant(1.0))}
    ev = events.at[:, 3].set(jnp.linspace(0, 1e-3, 5)) if Tlo != Thi else events.at[:, 1].set(jnp.linspace(0, 1e-3, 5))
    phi = np.asarray(line_profiles(ev, jnp.broadcast_to(jnp.array([1.0, 0, 0]), (5, 3)), v, 1.0, LineBundle(["HI-LyA"]), Fields({"HI": sp})))[0]
    for i in range(5):
        ax.semilogy(v, phi[i], color=list(TOL.values())[i])
    ax.set(xlabel=r"$v_\dagger$ [km/s]", ylabel=r"$\phi$ [ns]", ylim=(1e-8, None), title=title)
add_panel_letters(axs); fig.suptitle(r"Ly$\alpha$ profiles: temperature sets the width, the line-of-sight velocity the centre", fontsize=22); plt.show()

The detuning ν_j = c(1/λ′ − 1/λ_j), the photon's frequency offset from the line centre in the absorber's frame, uses the exact exponential mapping by default; `detuning="linear"` is fake_spectra's first-order version (matched mode, notebook 08). `velocity_period=P` folds the detuning into [−P/2, P/2) for periodic boxes.

## 3 · Optical depth: checks against closed forms

Static uniform gas (a = 1, no velocities) has closed forms (review notes): a Lorentzian for
T = 0 and a Gaussian for A_21 = 0 (the `null…` lines of the table). Lines of one species share the fields.

In [ ]:
Lk = 1e-3
def fields(T, n):
    sp = lambda: {"temperature": ScalarField(LogRamp(*T, Lk, 3)), "velocity": VectorField(VelocityRamp(0.0, 0.0, Lk)),
                  "number_density": ScalarField(LogRamp(*n, Lk, 2)), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
    return Fields({"HI": sp(), "SiIII": sp()})
null = LineBundle(["nullHI-LyA", "nullHI-LyB", "nullSiIII1206"])
pts = plane_points(Lk, "y", Lk / 2, 4, 32)                                       # 4 skewers, temperature rising with z
sb = SkewerBundle(spatial_points=pts, manual_scale_factors=jnp.ones_like(pts[None, ..., 0]))
v = jnp.linspace(-3000.0, 3000.0, 1024)
F_th = fields((1e4, 1e8), (1e-7, 1e-7))
tau = compute_optical_depth(sb, v, null, F_th, a_dagger=1.0)
T_sk = F_th["HI"]["temperature"](sb.quadri_vectors[0])[:, 0]
wl = line_wavelength_grid_A(v, null, 1.0, 1.0)
ref = jnp.stack([an.tau_static_thermal(wl[j], null.lambda0_A[j:j+1], null.mass_amu[j:j+1], null.f_osc[j:j+1], sb.a0, Lk * jnp.ones(4),
                                       1e-7 * jnp.ones(4), T_sk)[0] for j in range(3)])
F, Fa = np.exp(-np.asarray(tau)), np.exp(-np.asarray(ref))
fig, axs = plt.subplots(2, 3, figsize=(22, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
for j, l in enumerate(lines.lines):
    for k in range(4):
        c = list(TOL.values())[k]
        axs[0, j].plot(v, Fa[j, k], color=c, **MS["reference"])
        axs[0, j].plot(v, F[j, k], color=c, **MS["default"], label=f"T = {float(T_sk[k]):.0e} K" if j == 0 else None)
        axs[1, j].plot(v, F[j, k] - Fa[j, k], color=c)
    axs[0, j].set_title(l.name); axs[1, j].set_xlabel(r"$v_\dagger$ [km/s]")
axs[0, 0].set_ylabel("F"); axs[1, 0].set_ylabel(r"$F - F_{\rm analytic}$"); axs[0, 0].legend(fontsize=15)
add_panel_letters(axs); fig.suptitle("Static thermal lines: SkewerJAX (solid) on the analytic Gaussians (light); residuals < 10⁻⁸", fontsize=22); plt.show()

Radiation-dominated expansion and the convergence with the sampling are checked in `scripts/validation/check_optical_depth_analytic.py`.

**Expanding gas: the optical depth against the closed form.** Radiation-dominated expansion with uniform gas
(n0 = 5×10⁻¹¹ cm⁻³, A21 = 0, so the closed form is a difference of error functions; the review notes, `core.analytic.tau_radiation_thermal`), on two 5 cMpc skewers: one with the initial
conditions at the absorption end (a0 = 1), one at the emission end, at two temperatures.

In [ ]:
from skewerjax.core import Cosmology
from skewerjax.utils.plotting import plot_flux_vs_analytic
rad = Cosmology(H0=phys.H0, Omega_r_0=1.0, Omega_m_0=0.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
L5, n_pt, T0, n0 = 5.0, 336, jnp.array([5e4, 5e6]), 5e-11
sbr = SkewerBundle(spatial_points=plane_points(L5, "y", L5 / 2, 2, n_pt), a0=[1.0, 0.99999976], x0=[0.0, 0.0], ICs_defined_at_index=[-1, 0],
                   ICs_at_emission_or_absorption=["absorption", "emission"], cosmologies=rad)
sp_r = lambda: {"temperature": ScalarField(LogRamp(5e4, 5e6, L5, 3)), "velocity": VectorField(VelocityRamp(0.0, 0.0, L5)),
                "number_density": ScalarField(LogRamp(n0, n0, L5, 2)), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
v_r = jnp.linspace(-4000.0, 4000.0, 2048)
tau_r = compute_optical_depth(sbr, v_r, null, Fields({"HI": sp_r(), "SiIII": sp_r()}), a_dagger=jnp.repeat(jnp.asarray(sbr.a0)[:, None], n_pt, 1),
                              resolution_tolerance=None)
ref_r = jnp.stack([an.tau_radiation_thermal(line_wavelength_grid_A(v_r, null, 1.0, 1.0)[j], null.lambda0_A[j:j + 1], null.f_osc[j:j + 1],
                                            null.mass_amu[j:j + 1], T0, sbr.a0, sbr.x0, jnp.array([-L5, 0.0]), jnp.array([0.0, L5]),
                                            jnp.full(2, n0))[0] for j in range(3)])
a_far = np.asarray(sbr.scale_factors[0])[np.arange(2), np.where(np.asarray(sbr.ICs_defined_at_index) == 0, -1, 0)]
lam_obs = np.stack([np.stack([np.asarray(line_wavelength_grid_A(v_r, null, a0, a0))[j] for a0 in np.asarray(sbr.a0)]) for j in range(3)])
lam_c = np.asarray(null.lambda0_A)[:, None] * np.asarray(sbr.a0)[None] / a_far[None]
fig, _ = plot_flux_vs_analytic(np.asarray(v_r), np.exp(-np.asarray(tau_r)), np.exp(-np.asarray(ref_r)), null, lam_obs, lam_c, zoom_kms=600,
                               skewer_labels=[f"ICs at absorption, a0 = 1, T = 5e4 K", "ICs at emission, a0 = 0.99999976, T = 5e6 K"])
fig.suptitle("Radiation domination, thermal lines: numerical (solid) on the analytic F (light); zooms ±80 km/s and residuals", fontsize=21)
plt.show()
print("max |F - F_analytic| per line:", np.round(np.abs(np.exp(-np.asarray(tau_r)) - np.exp(-np.asarray(ref_r))).max(axis=(1, 2)), 5),
      f"({n_pt} points; second order in the spacing, check_optical_depth_analytic.py)")

*Reading the figure.* (a) On the observed-wavelength axis each line absorbs between its rest wavelength λ0
(dash-dot, the end of the path at a0) and λ0 a0/a_far (dashed), where the light left the other end of the box: the
Hubble flow across 5 cMpc spreads the absorption over a few hundred km/s, to the red or to the blue of λ0 depending
on where the initial conditions sit. (b–d) Around each line's v_† = 0 the numerical F lies on the closed form, and
(e–g) the residuals stay inside the ±10⁻² band; they fall as the square of the point spacing.

## 4 · Line-centred frames, the global velocity axis and the observed wavelength

Every line j is integrated on the same grid of **line-centred velocities** v_† (its own frame): the gas at a, moving
with v_∥ along the photon, absorbs the photon labelled v_† at λ′_j = λ_j e^{v_†/c} (a/a_†) D⁻¹. Its line centre sits
at the same v_c in every frame (`core.spectral.line_centre_velocity_kms`):

$$v_c = c\Big[\ln\frac{a_\dagger}{a} + \tfrac12\ln\frac{1-\beta}{1+\beta}\Big],\qquad \beta = v_\parallel/c .$$

The observer sees one axis. In observed wavelength, λ_obs = λ_j (a_obs/a_†) e^{v_†/c}
(`spectral.line_wavelength_grid_A`); on the **global velocity axis** of a reference line (here Lyα), the same pixel of
line j sits at v_Lyα = v_† − Δv_j with Δv_j = c ln(λ_Lyα/λ_j) (`LineBundle.delta_v`). `wrap_to_ref_line` performs that
shift on the uniform grid, treating it as periodic (one period of the box: what leaves one end comes back at the
other); without the wrap, the shifted spectrum simply extends beyond the grid.

The gas: four clumps of uniform-temperature gas (T = 10⁴ K, no peculiar velocities) along a 20 cMpc skewer at z ≈ 3,
Lyα and Si III λ1206 (Si III 5 times less abundant, to make it visible; both unsaturated, so the line centres can be read off). `compute_optical_depth` runs its three pure layers:
`los_samples` → `LoSSamples` (every field at every point, a pytree; the boundary used by `skewerjax.simulations`),
`optical_depth_from_samples` (jit; `lax.map` over skewers, `batch_size` of them vmapped, `v_chunk` velocities per call)
and `optical_depth_kernel` (one skewer); `resolution_report` says whether the points resolve the lines.

In [ ]:
from skewerjax.core import Cosmology, SampledField
from skewerjax.utils.plotting import plot_frames_axes
eds = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
L20, n_pt, z_obs = 20.0, 8193, 3.0
xs = np.linspace(0.0, L20, n_pt)
pts = jnp.asarray(np.stack([xs, np.full_like(xs, 1.0), np.full_like(xs, 1.0)], -1)[None])
sbf = SkewerBundle(spatial_points=pts, cosmologies=eds, a0=1 / (1 + z_obs), x0=L20)      # observer beyond the far end
clumps = [(3.0, 0.15), (7.5, 0.25), (12.0, 0.1), (16.5, 0.2)]                                  # (position, width) [cMpc]
n_x = 1.5e-12 * sum(np.exp(-0.5 * ((xs - c0) / w0) ** 2) for c0, w0 in clumps) + 1e-14
sp_c = lambda f: {"temperature": ScalarField(Constant(1e4)), "velocity": VectorField(VelocityRamp(0.0, 0.0, L20)),
                  "number_density": ScalarField(SampledField(jnp.asarray(f * n_x)[None])), "turbulence": ScalarField(Constant(0.0)),
                  "collisional": ScalarField(Constant(0.0))}
lya_si = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
a_pts = np.asarray(sbf.scale_factors[0, 0])
a_dag = 1 / (1 + z_obs)
P_box = float(phys.c_kms * np.log(a_pts.max() / a_pts.min()))                          # the path in velocity
v_box = (np.arange(2048) + 0.5) * P_box / 2048 + float(phys.c_kms * np.log(a_dag / a_pts.max()))
tau_f = compute_optical_depth(sbf, v_box, lya_si, Fields({"HI": sp_c(1.0), "SiIII": sp_c(1 / 5)}), a_dagger=a_dag)
v_cl = [float(phys.c_kms * np.log(a_dag / np.interp(c0, xs, a_pts))) for c0, _ in clumps]
fig, axs = plot_frames_axes(v_box, np.exp(-np.asarray(tau_f)[:, 0]), lya_si, a_dagger=a_dag, a_obs=1.0, centres=v_cl)
add_panel_letters(axs)
fig.suptitle("The same four clumps in observed wavelength (a), on the global Lyα axis (b) and in each line's own frame (c)", fontsize=21)
plt.show()
print(f"path: {P_box:.1f} km/s of Hubble flow; Delta v(Si III) = {float(lya_si.delta_v[1]):.1f} km/s = "
      f"{float(lya_si.delta_v[1]) / P_box:+.3f} periods; clump centres v_c = {np.round(v_cl, 1)} km/s (the same in both own frames)")

*Reading the figure.* (c) In its own frame each line absorbs at the clump's v_c (dotted): the same gas, the same
velocity, in every line. (a) On the observed-wavelength axis the two lines are 9.2 Å × (a_obs/a_†) apart. (b) On the
Lyα frame Si III moves by −Δv_j = −2271 km/s, to the blue (arrows: from the Lyα centre of each clump to its Si III). `wrap_to_ref_line` (solid) brings what leaves the grid back
in at the other end, as in a periodic box; the unwrapped shift (dashed) shows where the same absorption would sit on
an axis without a period. The wrap is exact only when τ itself is periodic on the grid (section 8).

## 5 · Option sweep: detuning and precision

The same moving gas (v_x from −300 to +300 km/s) with three settings of the τ kernel:
- the **exact detuning** (default): the frequency offset from the line centre in the absorber's frame,
  ν_j = (c/λ_j)[e^{−(v−v_c)/c} − 1], from the exact exponential mapping between v and wavelength;
- the **linear detuning** (`detuning="linear"`, fake_spectra's): ν_j = −(c/λ_j)(v − v_c)/c, the first-order expansion.
  The two differ by a relative (v − v_c)/(2c) in ν, so only far from the line centre (1.7×10⁻³ at 1000 km/s);
- the opt-in **single-precision kernel** (`dtype="float32"`; the result is returned in float64).

Where F = 0 (saturated) or F = 1 nothing can change, so (b) zooms into an unsaturated wing, chosen where the
options differ most, and (d) shows the relative change of τ itself.

In [ ]:
F_moving = Fields({sp: {"temperature": ScalarField(LogRamp(2e4, 2e4, Lk)), "velocity": VectorField(VelocityRamp(-300.0, 300.0, Lk)),
                         "number_density": ScalarField(LogRamp(1e-6, 1e-6, Lk)), "turbulence": ScalarField(Constant(0.0)),
                         "collisional": ScalarField(Constant(0.0))} for sp in ("HI", "SiIII")})
pts_x = jnp.stack([jnp.linspace(0, Lk, 1024), jnp.full(1024, Lk / 2), jnp.full(1024, Lk / 2)], -1)[None]   # along x: v_x ramps
for n_pt in (1024, 64):
    sbx = SkewerBundle(spatial_points=pts_x[:, ::1024 // n_pt], manual_scale_factors=jnp.ones((1, 1, n_pt)))
    rep = resolution_report(los_samples(sbx, F_moving, lya_si, a_dagger=1.0), lya_si, 0.25)
    print(f"{n_pt:4d} points: max |dv_c| / width per line = {np.round(rep['max_ratio'], 2)} -> {'resolved' if rep['ok'] else 'UNDER-RESOLVED (compute_optical_depth warns)'}")
sbx = SkewerBundle(spatial_points=pts_x, manual_scale_factors=jnp.ones((1, 1, 1024)))
v = jnp.linspace(-600, 2900, 1400)
tau = compute_optical_depth(sbx, v, lya_si, F_moving, a_dagger=1.0)
t_lin = compute_optical_depth(sbx, v, lya_si, F_moving, a_dagger=1.0, detuning="linear")
t_f32 = compute_optical_depth(sbx, v, lya_si, F_moving, a_dagger=1.0, dtype="float32")
from skewerjax.utils.plotting import plot_option_sweep
fig, axs, win = plot_option_sweep(np.asarray(v), {"exact (default)": np.asarray(tau)[:, 0], "linear detuning": np.asarray(t_lin)[:, 0],
                                                  "float32": np.asarray(t_f32)[:, 0]}, "exact (default)", [l.id for l in lya_si.lines])
fig.suptitle("What the detuning and precision options change, where it can be seen", fontsize=21); plt.show()
print(f"zoom window {win[0]:.0f} to {win[1]:.0f} km/s | max |dF|: linear detuning {np.abs(np.exp(-t_lin) - np.exp(-tau)).max():.1e}, "
      f"float32 {np.abs(np.exp(-t_f32) - np.exp(-tau)).max():.1e}")

## 6 · JAX: jit, vmap, grad

Every function above is pure; the containers are pytrees. For example, the derivative of the mean flux with respect
to the density of the gas:

In [ ]:
def mean_flux(n0):
    f = Fields({sp: {"temperature": ScalarField(LogRamp(2e4, 2e4, Lk)), "velocity": VectorField(VelocityRamp(-300.0, 300.0, Lk)),
                     "number_density": ScalarField(Constant(n0)), "turbulence": ScalarField(Constant(0.0)),
                     "collisional": ScalarField(Constant(0.0))} for sp in ("HI", "SiIII")})
    return jnp.exp(-compute_optical_depth(sbx, v, lya_si, f, a_dagger=1.0, resolution_tolerance=None)[0]).mean()
n0 = 1e-6
print(f"<F> = {float(mean_flux(n0)):.4f}, d<F>/d ln n = {float(jax.grad(mean_flux)(n0) * n0):.4f}, "
      f"finite difference {float((mean_flux(n0 * 1.001) - mean_flux(n0 * 0.999)) / 0.002):.4f}")

## 7 · τ from simulation outputs

The same integrator on the gas of notebook 05. `SimulationSkewers.optical_depth(lines)` integrates every line in its
own frame over one period of the box (1 km/s pixels by default), from the deposited pieces of every dual segment;
`fields(species)` gives the same gas as a `core.Fields` at the points.

In [ ]:
DATA = os.environ.get("SKEWERJAX_DATA", "/home/STORAGE")      # folder with TNG50-4/ (docs/installation.md)
from skewerjax.core import Cosmology
from skewerjax.simulations import ArepoHDF5Reader, SimulationSkewers, IntegrationOptions, HI
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
H = reader.header
cosmo = Cosmology(H0=100 * H.h * 1e6 * phys.Gyr_to_s / phys.Gpc_to_m, Omega_r_0=0.0, Omega_m_0=H.omega_m, Omega_k_0=0.0, Omega_lambda_0=H.omega_l)
L = H.box_cMpc
x = np.linspace(0.0, L, int(round(H.vmax_kms / 0.25)) + 1)
points = np.stack([np.stack([x, np.full_like(x, y), np.full_like(x, z)], -1) for y, z in [(10.0, 30.0), (25.0, 5.0), (40.0, 45.0)]])
bundle = SkewerBundle(spatial_points=jnp.asarray(points), a0=H.a, ICs_defined_at_index=0, ICs_at_emission_or_absorption="absorption", cosmologies=cosmo)
sk = SimulationSkewers.from_bundle(reader, bundle)
print(f"TNG50-4 snapshot 25: the three skewers of notebook 05, {len(sk.cells):,} cells selected in {sk.selection_s:.1f} s")

In [ ]:
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
velocities, tau = sk.optical_depth(lines)
flux = np.exp(-tau)
fields_sim = sk.fields(lines.species_keys)
sampled = fields_sim.sample(bundle.quadri_vectors[0], bundle.los_directions)
a_pts = np.asarray(bundle.quadri_vectors[0][..., 0])
v_pts = phys.c_kms * np.log(a_pts[:, :1] / a_pts)
fig, axs = plt.subplots(3, 3, figsize=(22, 12), sharex=True, sharey="row", layout="constrained")
for k in range(3):
    for j, l in enumerate(lines.lines):
        sty = MS["default"] if j == 0 else MS["matched"]
        axs[0, k].semilogy(v_pts[k], np.asarray(sampled[l.species]["number_density"])[k] / H.a ** 3, color=EC[l.species], **sty)
        axs[2, k].plot(velocities, flux[j, k], color=EC[l.species], **sty, label=l.id)
    axs[1, k].semilogy(v_pts[k], np.asarray(sampled["HI"]["temperature"])[k], color=EC["temperature"])
    axs[0, k].set_title(f"skewer {k}: y = {points[k, 0, 1]:.0f}, z = {points[k, 0, 2]:.0f} cMpc", fontsize=18); axs[2, k].set_xlabel("v [km/s]")
axs[0, 0].set_ylabel(r"$n$ [cm$^{-3}$]"); axs[1, 0].set_ylabel("T [K]"); axs[2, 0].set_ylabel("F"); axs[2, 0].legend()
add_panel_letters(axs, size=16); fig.suptitle("Three skewers through TNG50-4: the gas (top) and the flux it causes (bottom)", fontsize=22); plt.show()
print(f"<F>: Lya {flux[0].mean():.3f}, Si III {flux[1].mean():.4f}")

### Precision, zoom-in and devices

The τ of the three skewers again, with the opt-in float32 kernel (`IntegrationOptions(dtype="float32")`). The
float64 default is the reference. The inset zooms into one absorber. The same calls take `devices=jax.devices()`
to shard the work over several devices (notebook 08 and `docs/usage.md`).

In [ ]:
import time
t0 = time.perf_counter(); _, tau64 = sk.optical_depth(lines); t64 = time.perf_counter() - t0
t0 = time.perf_counter(); _, tau32 = sk.optical_depth(lines, integ=IntegrationOptions(dtype="float32")); t32 = time.perf_counter() - t0
fig, axs = plt.subplots(2, 1, figsize=FIGSIZES["tall"], sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
plot_spectra(axs[0], velocities, np.exp(-tau64[:, 0]), [l.id for l in lines.lines], total=False)
for j, l in enumerate(lines.lines):
    axs[1].plot(velocities, np.exp(-tau32[j, 0]) - np.exp(-tau64[j, 0]), color=EC[l.id], lw=1.2, label=l.id)
k0 = int(np.argmin(np.exp(-tau64[0, 0])))
zoom_inset(axs[0], (velocities[k0] - 60, velocities[k0] + 60), (-0.02, 1.02), bounds=(0.66, 0.1, 0.32, 0.5))
axs[1].set(xlabel=LABELS["v_dagger"], ylabel=r"$F_{32} - F_{64}$"); axs[0].legend(); axs[1].legend()
add_panel_letters(axs); fig.suptitle(f"float32 kernel: max |ΔF| = {np.abs(np.exp(-tau32) - np.exp(-tau64)).max():.1e}, "
                                     f"{t64:.1f} s → {t32:.1f} s (first calls include compilation)", fontsize=20); plt.show()

## 8 · How the integration is performed, and where periodicity enters

**The integral.** For every line j, τ_j(v) = 𝒞 f_j Σ_i w_i n_com,i a_i⁻² φ_j(ν_j(v; a_i, v_∥,i)): a sum over the
sampling points (core) or over the deposited pieces (simulations), each with its own line centre. Nothing in this
sum is periodic. Periodicity can enter in four places:

| where | core (`compute_optical_depth`) | simulations (`SimulationSkewers`) |
|---|---|---|
| the gas at the points | toy samplers wrap positions into the box (`GaussianDensity`, periodic interpolation): a path longer than the box sees the same gas again | the deposition takes every cell at its minimum image: paths may cross the box faces |
| the detuning | **default: none** (`velocity_period=None`, the direct sum); `velocity_period=P` replaces v − v_c by its nearest periodic image in [−P/2, P/2) | **default: periodic** (`IntegrationOptions(periodic=True)`, P = the box in velocity, c ln(a_0/a_L)); `periodic=False` gives the direct sum |
| the spectral grid | any grid | `spectral_grid`: exactly one period, n_bins = int(P/res) |
| one axis for all lines | `wrap_to_ref_line`: always a periodic shift on the grid, period N_v dv | the same |

So the simulation spectra of this notebook are periodic by construction (each detuning wrapped, a grid of one period),
while a core integral is the direct sum of the finite path unless `velocity_period` is given. The figure compares the
two on the toy box of notebook 00 (50 cMpc, z = 2.2, one GRF for H I and Si III), on a grid of exactly one period:
(a) they differ only near the ends of the grid, where the direct sum loses the absorption of gas whose line spills
over the box edge; (b) on the periodic integral the wrap is exact: Si III moved by `wrap_to_ref_line` equals Si III
evaluated directly at v + Δv in its own frame (up to the linear interpolation between pixels); (c) wrapping the direct
integral moves its cut edge to the middle of the reference-frame spectrum (dotted: −Δv mod P).

In [ ]:
from skewerjax.core import Cosmology, grid_skewer_points, PowerLawGRF, GaussianDensity
eds = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
Lb, z = 50.0, 2.2
pts = grid_skewer_points(Lb, 2, 16385, direction="y").reshape(4, 16385, 3).astype(jnp.float64)   # first and last point: the same place
sb4 = SkewerBundle(spatial_points=pts, cosmologies=eds, a0=1 / (1 + z), x0=Lb)
a_d = 1 / (1 + z)
P = float(phys.c_kms * jnp.log(a_d / sb4.scale_factors[0, 0, 0]))             # the box in velocity: one period [km/s]
vq = (np.arange(2048) + 0.5) * P / 2048                                         # one period, N_v dv = P exactly
grf_q = PowerLawGRF(A_s=4.0, k_0=2.0, n_s=-1.25, L=(Lb,) * 3, N=(64,) * 3)
key_q = jax.random.PRNGKey(0)
toy_sp = lambda d: {"temperature": ScalarField(LogRamp(1.5e4, 1.5e4, Lb)), "velocity": VectorField(VelocityRamp(0.0, 0.0, Lb)),
                    "number_density": ScalarField(d), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
dens_q = {"HI": GaussianDensity(key_q, grf_q, n0_min=1e-16, n0_max=1.35e-11, contrast=1.2),
          "SiIII": GaussianDensity(key_q, grf_q, n0_min=1e-16, n0_max=1.35e-11 / 250, contrast=1.2)}
toy = Fields({sp: toy_sp(d) for sp, d in dens_q.items()})
dV = float(lya_si.delta_v[1])
tau_per = compute_optical_depth(sb4, vq, lya_si, toy, a_dagger=a_d, velocity_period=P)          # periodic detuning
tau_dir = compute_optical_depth(sb4, vq, lya_si, toy, a_dagger=a_d)                             # direct sum (default)
si_only = LineBundle(["SiIII-1206"], ref_line_id="SiIII-1206")
tau_si_shift = compute_optical_depth(sb4, vq + dV, si_only, Fields({"SiIII": toy_sp(dens_q["SiIII"])}), a_dagger=a_d, velocity_period=P)[0]
F_per, F_dir = np.exp(-np.asarray(tau_per)), np.exp(-np.asarray(tau_dir))
F_ref_per = np.exp(-np.asarray(wrap_to_ref_line(tau_per, vq, lya_si)))
F_ref_dir = np.exp(-np.asarray(wrap_to_ref_line(tau_dir, vq, lya_si)))
fig, axs = plt.subplots(1, 3, figsize=FIGSIZES["triple"], layout="constrained")
for k in range(4):
    c = list(TOL.values())[k]
    axs[0].plot(vq, F_dir[0, k] - F_per[0, k], color=c, label=f"skewer {k}")
    axs[1].plot(vq, F_ref_per[1, k] - np.exp(-np.asarray(tau_si_shift[k])), color=c)
    axs[2].plot(vq, F_ref_dir[1, k] - F_ref_per[1, k], color=c)
axs[0].set(xlabel=LABELS["v_dagger"], ylabel=r"$F_{\rm direct} - F_{\rm periodic}$ (Ly$\alpha$)", title="(a) direct vs periodic detuning")
axs[1].set(xlabel=LABELS["v_ref"], ylabel=r"$\Delta F$ (Si III)", title=r"(b) wrap vs direct evaluation at $v + \Delta v$")
axs[2].set(xlabel=LABELS["v_ref"], ylabel=r"$\Delta F$ (Si III)", title="(c) wrap of the direct sum − wrap of the periodic one")
axs[2].axvline(np.mod(-dV, P), color=FG, ls=":", lw=1.2)
axs[0].legend(); plt.show()
print(f"P = {P:.1f} km/s | max |dF|: (a) {np.abs(F_dir - F_per).max():.2e} | (b) {np.abs(F_ref_per[1] - np.exp(-np.asarray(tau_si_shift))).max():.2e} "
      f"| (c) {np.abs(F_ref_dir[1] - F_ref_per[1]).max():.2e}")

**Choosing a mode.** A periodic integral is right for a periodic simulation box when the spectrum covers exactly one
period, which is what P1D on snapshots needs. The direct sum is right for a path that is not a box: a light cone, a
curved path, or a segment of a longer sightline. The alternatives (a buffer of extra path at both ends, or a sum
over periodic images) are not implemented. In every case the shift onto a reference line is periodic on the grid,
so it is exact only for a periodic τ.